In [0]:
from pyspark.sql.functions import col, to_date, sum, round

In [0]:
catalog = "medallion_trial"

In [0]:
print("Starting Gold Layer Aggregation...")

# 1. Read the cleaned Silver tables
df_orders = spark.table(f"{catalog}.silver.cleaned_orders")
df_items = spark.table(f"{catalog}.silver.cleaned_order_items")

# 2. Join the tables together on 'order_id'
df_joined = df_orders.join(df_items, on="order_id", how="inner")

# 3. Calculate Daily Revenue
# Convert timestamp to a simple date, group by it, and sum the prices
df_daily_revenue = df_joined \
    .withColumn("order_date", to_date(col("order_purchase_timestamp"))) \
    .groupBy("order_date") \
    .agg(
        round(sum("price"), 2).alias("total_revenue_usd"),
        round(sum("freight_value"), 2).alias("total_freight_usd")
    ) \
    .orderBy("order_date")

# 4. Write to the Gold schema as a Delta table
df_daily_revenue.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.daily_revenue")

print("✅ Daily Revenue table successfully saved to Gold!")

In [0]:
%sql
SELECT * FROM medallion_trial.gold.daily_revenue 
WHERE order_date IS NOT NULL 
ORDER BY total_revenue_usd DESC 
LIMIT 10;